In [ ]:
from collections import defaultdict # access key with undefined value
import matplotlib.pyplot as plt # graph
import matplotlib.patches as mpatches # draw shapes
import seaborn as sns # draw pandas dataframe
import numpy as np # data and processing

from tqdm import tqdm # to understand training progress

import gymnasium as gym # training environment

In [ ]:
env = gym.make("Blackjack-v1", sab=True, render_mode="rgb_array") 

In [ ]:
# done = False
# state, info = env.reset() 

In [ ]:
# action = env.action_space.sample() 
# observation, reward, terminated, truncated, info = env.step(action)

In [ ]:
class BlackjackAgent:
	def __init__(
	self, env:gym.Env,
	learning_rate:float =0.1, 
	initial_epsilon:float =1.0, 
	epsilon_decay:float =0.999,
	final_epsilon:float =0.1,
	discount_factor:float =0.5,
	):
		self.learning_rate = learning_rate
		self.epsilon = initial_epsilon
		self.epsilon_decay = epsilon_decay
		self.final_epsilon = final_epsilon
		self.discount_factor = discount_factor
		self.q_table = defaultdict(lambda: np.zeros(env.action_space.n)) # action space is 2, hit or stick
		self.training_error = []

	def get_action(self, state:tuple[int, int, bool]) -> int:
		if np.random.rand() < self.epsilon:
			return env.action_space.sample() # explore
		else:
			return np.argmax(self.q_table[state]) # exploit
	
	def update_q_value(
			self, 
			state:tuple[int, int, bool], 
			action:int, 
			reward:float,
			terminated:bool, 
			next_state:tuple[int, int, bool],
			) -> None:
		future_q_value = (not terminated) * np.max(self.q_table[next_state]) 
		temporal_difference = reward + self.discount_factor * future_q_value - self.q_table[state][action]
		self.q_table[state][action] += self.learning_rate * temporal_difference

		
		self.training_error.append(temporal_difference)

	def decay_epsilon(self) -> None:
		self.epsilon = max(self.final_epsilon, self.epsilon * self.epsilon_decay)

In [ ]:
learning_rate = 0.01
n_episodes = 10_000
start_epsilon = 1.0
epsilon_decay = 2 * start_epsilon / (n_episodes) # same as start_epsilon / (n_episodes / 2)
final_epsilon = 0.1

agent = BlackjackAgent(
	env=env,
	learning_rate=learning_rate,
	initial_epsilon=start_epsilon,
	epsilon_decay=epsilon_decay,
	final_epsilon=final_epsilon,
	discount_factor=0.5,
	)

In [ ]:
from IPython.display import display, clear_output

env = gym.wrappers.RecordEpisodeStatistics(env, buffer_length=n_episodes)

for episode in tqdm(range(n_episodes), desc="Training"):
	state, info = env.reset()
	done = False
	clear_output(wait=True)
	while not done:
		action = agent.get_action(state)
		next_state, reward, terminated, truncated, info = env.step(action)

		agent.update_q_value(state, action, reward, terminated, next_state)
		frame=env.render()
		plt.imshow(frame)
		plt.show()

		done = terminated or truncated
		state = next_state
		
	agent.decay_epsilon()

In [ ]:
rolling_length = 500
fig, ax = plt.subplots(ncols=3 ,figsize=(12, 5))
ax[0].title.set_text("Episode Rewards")
rewards_moving_average = np.convolve(env.return_queue, np.ones(rolling_length)/rolling_length, mode='valid')/rolling_length
ax[0].plot(range(len(rewards_moving_average)), rewards_moving_average)
ax[1].title.set_text("Episode Lengths")
lengths_moving_average = np.convolve(env.length_queue, np.ones(rolling_length)/rolling_length, mode='valid')/rolling_length
ax[1].plot(range(len(lengths_moving_average)), lengths_moving_average)
ax[2].title.set_text("Training Error")
training_error_moving_average = np.convolve(agent.training_error, np.ones(rolling_length)/rolling_length, mode='valid')/rolling_length
ax[2].plot(range(len(training_error_moving_average)), training_error_moving_average)
plt.tight_layout()
plt.show()